# Model Evaluation & Metrics

## Classification Metrics

For classification, the confusion matrix is the foundation: True Positives (TP), True Negatives (TN), False Positives (FP), False Negatives (FN).

- **Accuracy** = (TP + TN) / (TP + TN + FP + FN) — Overall correctness (misleading for imbalanced data)
- **Precision** = TP / (TP + FP) — Of predicted positives, how many are correct? (minimize false alarms)
- **Recall** = TP / (TP + FN) — Of actual positives, how many did we catch? (minimize missed cases)
- **F1-score** = 2 × (Precision × Recall) / (Precision + Recall) — Harmonic mean (balanced metric)

```python title="example1.py"
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Load data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")
print(f"TP: {cm[1,1]}, FP: {cm[0,1]}, FN: {cm[1,0]}, TN: {cm[0,0]}")

# Classification report
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# Precision-Recall curve (for threshold tuning)
y_proba = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Find threshold maximizing F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Optimal threshold: {best_threshold:.3f}, F1: {f1_scores[best_idx]:.3f}")
```

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/scikit-learn-machine-learning/mod-27.ipynb)

```
              precision    recall  f1-score   support
    Negative       0.92      0.95      0.93       150
    Positive       0.88      0.82      0.85        50
    accuracy                           0.91       200
```

## ROC-AUC & Threshold Selection

ROC-AUC (Receiver Operating Characteristic - Area Under Curve) measures the model's ability to distinguish between classes across all thresholds. AUC = 1.0 is perfect; 0.5 is random.

```python title="example2.py"
from sklearn.metrics import roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

# For imbalanced data, use PR-AUC instead
from sklearn.metrics import average_precision_score
pr_auc = average_precision_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.3f}, PR-AUC: {pr_auc:.3f}")
```

> **💡 Tip:** For imbalanced datasets, use PR-AUC (Precision-Recall AUC) instead of ROC-AUC. ROC-AUC can be misleading when one class dominates.

## Regression Metrics

For regression, common metrics are:
- **MAE** (Mean Absolute Error) — Average absolute difference
- **RMSE** (Root Mean Squared Error) — Penalizes large errors more
- **R²** (Coefficient of Determination) — Proportion of variance explained (0-1)

```python title="example3.py"
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predictions
y_pred_reg = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_reg))
r2 = r2_score(y_test, y_pred_reg)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")
```

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ When should you use Recall over Precision?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700001" value="0">
      <span>When false positives are costly (spam detection)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700001" value="1">
      <span>When the dataset is balanced</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700001" value="2">
      <span>When false negatives are costly (disease detection)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700001" value="3">
      <span>When training time is limited</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What does ROC-AUC = 0.5 indicate?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700002" value="0">
      <span>Perfect classification</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700002" value="1">
      <span>Random guessing (no discriminative power)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700002" value="2">
      <span>50% accuracy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387700002" value="3">
      <span>Balanced precision and recall</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>